# Notebook 09 -- Brand Benchmarking Scorecard

## 1. Load Data

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Brand Distribution

In [2]:
df = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
print(f"Total reviews: {len(df):,}")
display(df["brand_category"].value_counts().to_frame())

Total reviews: 381,999


,count
brand_category,
Independent Café,362334
Starbucks,11675
Dunkin',6866
Dutch Bros,925
The Coffee Bean,128
Peet's Coffee,71


## 3. Brand-Level Summary Metrics

In [3]:
scored_brands = ["Starbucks", "Dunkin'", "Independent Café"]

rows = []
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rows.append({
        "Brand":         b,
        "Reviews":       len(g),
        "Avg Stars":     round(g["review_stars"].mean(), 2),
        "% Positive":    round((g["star_tier"] == "Positive").mean() * 100, 1),
        "% Critical":    round((g["star_tier"] == "Critical").mean() * 100, 1),
        "Avg Sentiment": round(g["sentiment_score"].mean(), 3),
    })

tbl = pd.DataFrame(rows)
display(tbl)

,Brand,Reviews,Avg Stars,% Positive,% Critical,Avg Sentiment
0,Starbucks,11675,2.90,42.4,47.6,0.277
1,Dunkin',6866,2.05,21.5,71.2,-0.020
2,Independent Café,362334,4.01,73.7,17.2,0.706


## 4. Topic-Level Positive Rates

In [4]:
topic_dims = ["Service", "Food Quality", "Ambiance", "Wait Time", "Cleanliness"]

scores = {}
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rates = {}
    for t in topic_dims:
        # Multi-label: count reviews that mention this topic (may also carry other tags)
        mask  = g["topic_tags"].str.contains(t, regex=False)
        total = mask.sum()
        pos   = (mask & (g["star_tier"] == "Positive")).sum()
        rates[t] = round(pos / total * 100, 1) if total > 0 else 0
    scores[b] = rates

score_df = pd.DataFrame(scores).T
score_df["Composite"] = score_df.mean(axis=1).round(1)
display(score_df)

,Service,Food Quality,Ambiance,Wait Time,Cleanliness,Composite
Starbucks,44.4,39.5,62.2,39.9,47.2,46.6
Dunkin',22.1,22.2,42.2,25.2,27.2,27.8
Independent Café,71.5,73.9,78.8,65.4,62.4,70.4


## 5. Radar Chart

In [5]:
display(score_df[topic_dims])

# close the polygon for each trace
theta = topic_dims + [topic_dims[0]]

brand_styles = {
    "Starbucks":        {"color": "#00704A", "fill": "rgba(0,112,74,0.25)",  "dash": "solid"},
    "Dunkin'":          {"color": "#E04006", "fill": "rgba(224,64,6,0.20)",  "dash": "dash"},
    "Independent Café": {"color": "#1f77b4", "fill": "rgba(31,119,180,0.15)", "dash": "dot"},
}

fig = go.Figure()
for brand, style in brand_styles.items():
    vals = [scores[brand][d] for d in topic_dims] + [scores[brand][topic_dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals,
        theta=theta,
        name=brand,
        mode="lines+markers",
        line=dict(color=style["color"], dash=style["dash"], width=2.5),
        marker=dict(color=style["color"], size=8),
        fill="toself",
        fillcolor=style["fill"],
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100],
            tickvals=[0, 25, 50, 75, 100],
            ticksuffix="%",
            gridcolor="#dddddd",
            linecolor="#dddddd",
        ),
        angularaxis=dict(
            gridcolor="#dddddd",
            linecolor="#dddddd",
        ),
    ),
    title=dict(
        text="Brand Scorecard — Topic Positive Rate (%) by Brand",
        font=dict(size=16),
        x=0.5,
    ),
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    paper_bgcolor="white",
    width=650, height=600,
    margin=dict(t=80, b=90, l=60, r=60),
)
fig.write_html(str(FIGURES_DIR / "09_scorecard_rose.html"))
fig.show()

,Service,Food Quality,Ambiance,Wait Time,Cleanliness
Starbucks,44.4,39.5,62.2,39.9,47.2
Dunkin',22.1,22.2,42.2,25.2,27.2
Independent Café,71.5,73.9,78.8,65.4,62.4


## Key Findings

- Independent cafes average 4.01 stars (73.7% positive). Starbucks sits at 2.90 (42.4% positive) and Dunkin' at 2.05 (21.5% positive).
- Dunkin' scores lowest on every topic dimension. Composite score: 27.8 vs. Starbucks 46.6 and Independents 70.4.
- Ambiance is the strongest topic for chains. Starbucks scores 62.2% and Dunkin' 42.2%, each brand's highest positive rate.
- Food Quality is Starbucks' weakest area at 39.5% positive, compared to 73.9% for independents. That is the widest single-topic gap.